In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
import matplotlib.pyplot as plt

print("🚀 Εκκίνηση XGBoost με τα αρχεία .csv...")

# --- 1. ΦΟΡΤΩΣΗ EMBEDDINGS (ΤΑ ΕΝΩΝΟΥΜΕ) ---
print("   ... Φόρτωση Embeddings...")
try:
    # Φορτώνουμε τα δύο κομμάτια
    emb1 = np.load('esm2_embeddings_1.npy')
    emb2 = np.load('esm2_embeddings_2.npy')

    # Τα ενώνουμε (Concatenation)
    embeddings = np.concatenate((emb1, emb2), axis=0)
    print(f"✅ Embeddings loaded & merged. Shape: {embeddings.shape}")
except Exception as e:
    print(f"❌ Σφάλμα στη φόρτωση των embeddings: {e}")
    print("👉 Βεβαιώσου ότι ανέβασες τα esm2_embeddings_1.npy και _2.npy")

# --- 2. ΦΟΡΤΩΣΗ DATASETS (.CSV) ---
print("   ... Φόρτωση CSV αρχείων...")
try:
    # Διαβάζουμε τα CSV
    # Αν βγάλει λάθος, ίσως χρειάζεται header=None, αλλά συνήθως έχουν επικεφαλίδες
    df_pos = pd.read_csv('positive_protein_sequences.csv')
    df_pos['label'] = 1

    df_neg = pd.read_csv('negative_protein_sequences.csv')
    df_neg['label'] = 0

    # Ένωση (Concatenation) των δύο σε ένα μεγάλο DataFrame
    df = pd.concat([df_pos, df_neg], ignore_index=True)

    print(f"✅ Dataset loaded. Total pairs: {len(df)}")
    print(f"   Positives: {len(df_pos)}, Negatives: {len(df_neg)}")

    # Τυπώνουμε τις πρώτες γραμμές για να δούμε αν διαβάστηκαν σωστά οι στήλες
    print("   Στήλες που βρέθηκαν:", df.columns.tolist())

except Exception as e:
    print(f"❌ Σφάλμα στη φόρτωση των CSV: {e}")

# --- 3. ΑΝΤΙΣΤΟΙΧΙΣΗ (MAPPING) ---
print("   ... Δημιουργία Mapping Πρωτεϊνών...")

# Βρίσκουμε τις στήλες (συνήθως οι 2 πρώτες είναι οι πρωτεΐνες)
col_A = df.columns[0]
col_B = df.columns[1]

# Βρίσκουμε όλες τις μοναδικές πρωτεΐνες στο dataset και τις ταξινομούμε Αλφαβητικά
# (Αυτό είναι κρίσιμο γιατί έτσι φτιάχτηκαν τα embeddings αρχικά)
unique_proteins = sorted(list(set(df[col_A].unique()) | set(df[col_B].unique())))

print(f"   Unique Proteins found in CSV: {len(unique_proteins)}")
print(f"   Embeddings available: {embeddings.shape[0]}")

# Έλεγχος αν ταιριάζουν
if len(unique_proteins) != embeddings.shape[0]:
    print("⚠️ ΠΡΟΣΟΧΗ: Ασυμφωνία αριθμού πρωτεϊνών!")
else:
    print("✅ Τέλεια αντιστοίχιση!")

# Λεξικό: Όνομα Πρωτεΐνης -> Αριθμός Γραμμής στο Embedding
prot_to_idx = {prot: i for i, prot in enumerate(unique_proteins)}

# --- 4. ΠΡΟΕΤΟΙΜΑΣΙΑ ΧΑΡΑΚΤΗΡΙΣΤΙΚΩΝ (FEATURES) ---
print("⚙️ Προετοιμασία πινάκων για XGBoost...")

# Λίστες με τα indices για γρήγορη ανάκτηση
idx_A = [prot_to_idx[p] for p in df[col_A]]
idx_B = [prot_to_idx[p] for p in df[col_B]]

# Ανάκτηση embeddings από τον πίνακα
emb_A = embeddings[idx_A]
emb_B = embeddings[idx_B]

# Ένωση (Concatenation): Το feature κάθε ζεύγους είναι [Vector A + Vector B]
X = np.concatenate([emb_A, emb_B], axis=1)
y = df['label'].values

print(f"✅ Feature Matrix X shape: {X.shape}")

# --- 5. ΕΚΠΑΙΔΕΥΣΗ XGBOOST ---
print("🔥 Εκπαίδευση XGBoost Classifier (Παρακαλώ περιμένετε)...")

# Split σε Train (80%) και Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Ρυθμίσεις XGBoost
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    tree_method='hist', # Γρήγορη μέθοδος
    random_state=42
)

model.fit(X_train, y_train)

# --- 6. ΑΠΟΤΕΛΕΣΜΑΤΑ ---
preds = model.predict(X_test)
probs = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, preds)
auc = roc_auc_score(y_test, probs)

print("\n" + "="*40)
print(f"🏆 XGBoost Accuracy: {acc*100:.2f}%")
print(f"📊 XGBoost ROC-AUC: {auc:.4f}")
print("="*40)

# Feature Importance Plot
plt.figure(figsize=(10, 6))
xgb.plot_importance(model, max_num_features=15, height=0.5, importance_type='weight', title='XGBoost Feature Importance')
plt.show()

In [ ]:
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc

# Ρυθμίσεις για ωραία γραφήματα
plt.figure(figsize=(14, 6))

# --- 1. Confusion Matrix ---
plt.subplot(1, 2, 1)
cm = confusion_matrix(y_test, preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Interaction', 'Interaction'],
            yticklabels=['No Interaction', 'Interaction'])
plt.title('XGBoost Confusion Matrix', fontsize=14)
plt.ylabel('Πραγματική Τιμή (True Label)')
plt.xlabel('Πρόβλεψη Μοντέλου (Predicted Label)')

# --- 2. ROC Curve ---
plt.subplot(1, 2, 2)
fpr, tpr, thresholds = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'XGBoost (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--') # Διαγώνιος (Random Guess)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.title('Receiver Operating Characteristic (ROC)', fontsize=14)
plt.legend(loc="lower right")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()